# Assignment Part 6 — Isolation Testing (Concurrency)

Proves the database prevents race conditions via two tests:

**Test 1 — Double-Booking Prevention:**
50 threads simultaneously attempt to book OR-1 at exactly the same time slot.
Expected result: exactly 1 succeeds, 49 get `RoomConflictError`.

**Test 2 — GIST Constraint Safety Net:**
Attempt a raw SQL INSERT that bypasses application logic to double-book a room.
Expected: PostgreSQL's GIST exclusion constraint rejects it with `IntegrityError`.

## How It Works
PostgreSQL's `SELECT FOR UPDATE` acquires an exclusive row-level lock on the Room row.
When 50 threads race to acquire the same lock, they **serialize** — only one proceeds
at a time. The first thread to commit creates the reservation. All subsequent threads
re-check the reservation table and find a conflict → `RoomConflictError`.

```
Thread-A: SELECT room FOR UPDATE  → acquires lock
Thread-B: SELECT room FOR UPDATE  → BLOCKED (waits for A's lock)
Thread-A: INSERT reservation      
Thread-A: COMMIT                  → lock released
Thread-B: lock granted, re-reads  → sees A's reservation → RoomConflictError
```

In [17]:
import sys
sys.path.insert(0, '../src')

import threading
import time as time_module
from datetime import date, time, timedelta, datetime, timezone
from rich.console import Console
from rich.table import Table
from rich import print as rprint
console = Console()

from sqlalchemy import select, text
from sqlalchemy.orm import Session
from sqlalchemy.exc import IntegrityError
from or_scheduler.database import engine, SessionLocal
from or_scheduler.models import Department, Staff, Room, Patient, RoomReservation, Appointment
from or_scheduler.operations import (
    create_case, create_appointment, StaffItem, RoomConflictError
)

## Setup

In [18]:
ISOLATION_DATE = date.today() + timedelta(days=30)  # fresh date, no existing reservations
SLOT_START = time(8, 0)
SLOT_END = time(10, 0)

with Session(engine) as s:
    dept = s.execute(select(Department).limit(1)).scalar_one()
    surgeon = s.execute(select(Staff).where(Staff.role == 'SURGEON').limit(1)).scalar_one()
    anaest = s.execute(select(Staff).where(Staff.role == 'ANAESTHESIOLOGIST').limit(1)).scalar_one()
    room = s.execute(select(Room).where(Room.room_code == 'OR-1')).scalar_one()
    patients = s.execute(select(Patient).limit(60)).scalars().all()
    
    dept_id = dept.department_id
    surgeon_id = surgeon.staff_id
    anaest_id = anaest.staff_id
    room_id = room.room_id
    patient_hns = [p.hn for p in patients]

print(f"Target room: {room.room_code}")
print(f"Target date: {ISOLATION_DATE}")
print(f"Target slot: {SLOT_START}–{SLOT_END}")

Target room: OR-1
Target date: 2026-04-03
Target slot: 08:00:00–10:00:00


In [19]:
# Isolation test idempotency — clean up any OR-1 reservations from previous runs
from or_scheduler.models import RoomSchedule, StaffSchedule

print(f"Cleaning up stale isolation test data for {ISOLATION_DATE}...")
with SessionLocal() as _s:
    stale_ids = [str(r[0]) for r in _s.execute(text("""
        SELECT a.appointment_id FROM appointments a
        JOIN room_reservations rr ON rr.appointment_id = a.appointment_id
        WHERE rr.room_id = :rid AND a.scheduled_date = :d
    """), {'rid': str(room_id), 'd': ISOLATION_DATE}).fetchall()]

    if stale_ids:
        _s.execute(text("DELETE FROM equipment_reservations WHERE appointment_id::text = ANY(:ids)"), {'ids': stale_ids})
        _s.execute(text("DELETE FROM staff_reservations WHERE appointment_id::text = ANY(:ids)"), {'ids': stale_ids})
        _s.execute(text("DELETE FROM room_reservations WHERE appointment_id::text = ANY(:ids)"), {'ids': stale_ids})
        _s.execute(text("DELETE FROM appointments WHERE appointment_id::text = ANY(:ids)"), {'ids': stale_ids})
        _s.commit()
        print(f"  Removed {len(stale_ids)} stale appointment(s) — clean slate")
    else:
        print(f"  No stale data — already clean")

# Ensure schedules exist for the isolation test date
with SessionLocal() as s:
    for resource_id, model, kwarg in [
        (room_id, RoomSchedule, {'room_id': room_id}),
        (surgeon_id, StaffSchedule, {'staff_id': surgeon_id}),
        (anaest_id, StaffSchedule, {'staff_id': anaest_id}),
    ]:
        if model == RoomSchedule:
            existing = s.execute(select(model).where(
                model.room_id == room_id, model.date == ISOLATION_DATE
            )).scalar_one_or_none()
        else:
            existing = s.execute(select(model).where(
                model.staff_id == resource_id, model.date == ISOLATION_DATE
            )).scalar_one_or_none()
        if not existing:
            s.add(model(**kwarg, date=ISOLATION_DATE,
                        available_from=time(8,0), available_until=time(17,0),
                        schedule_type='REGULAR'))
    s.commit()

# Pre-create 50 cases (one per thread)
print("Pre-creating 50 cases for isolation test...")
case_ids = []
with SessionLocal() as s:
    for hn in patient_hns[:50]:
        cr = create_case(s, patient_hn=hn, department_id=dept_id,
                         surgeon_id=surgeon_id,
                         procedure_type='Isolation Test Procedure',
                         created_by=surgeon_id)
        case_ids.append(cr.case_id)
    s.commit()
print(f"✓ {len(case_ids)} cases ready")

Cleaning up stale isolation test data for 2026-04-03...
  Removed 4 stale appointment(s) — clean slate
Pre-creating 50 cases for isolation test...
✓ 50 cases ready


## Test 1 — Race Condition (Part A: The Problem Without Locking)

**Naive check-then-insert pattern** — no `SELECT FOR UPDATE`:

```
Thread A: SELECT COUNT(*) = 0  ← room appears free (no lock held)
Thread B: SELECT COUNT(*) = 0  ← room appears free (same snapshot!)
          ... 20ms passes ...
Thread A: INSERT booking        ← succeeds
Thread B: INSERT booking        ← also succeeds! (no constraint to stop it)
Result  : 2+ rows for the same OR slot — DOUBLE BOOKING
```

The cells below use an **unprotected temporary table** (no GIST, no UNIQUE constraints) to produce actual data corruption, proving the race condition is real — not theoretical.

In [20]:
import threading as _threading

# Create a persistent table with NO constraints — simulates an unprotected system
# Must be a regular table (NOT TEMP): PostgreSQL TEMP tables are connection-scoped,
# so the spawned threads (each on a different pool connection) would not see a TEMP table.
with engine.connect() as _conn:
    _conn.execute(text("DROP TABLE IF EXISTS naive_bookings"))
    _conn.execute(text("""
        CREATE TABLE naive_bookings (
            booking_id   uuid DEFAULT gen_random_uuid() PRIMARY KEY,
            room_code    text        NOT NULL,
            booked_from  timestamptz NOT NULL,
            booked_until timestamptz NOT NULL,
            thread_id    int         NOT NULL
        )
    """))
    _conn.commit()

_start_dt = datetime.combine(ISOLATION_DATE, SLOT_START, tzinfo=timezone.utc)
_end_dt   = datetime.combine(ISOLATION_DATE, SLOT_END,   tzinfo=timezone.utc)

def naive_book(thread_idx: int):
    """Check-then-insert WITHOUT any lock — demonstrates the race condition vulnerability."""
    with engine.connect() as _c:
        # Step 1: Read — check if room is free (NO LOCK ACQUIRED)
        existing = _c.execute(text("""
            SELECT COUNT(*) FROM naive_bookings
            WHERE room_code  = 'OR-1'
              AND booked_from  < :end_dt
              AND booked_until > :start_dt
        """), {'start_dt': _start_dt, 'end_dt': _end_dt}).scalar()

        if existing > 0:
            return ('conflict', thread_idx)

        # ── RACE WINDOW ──────────────────────────────────────────────────────
        # All other threads can also read "room is free" here —
        # no lock prevents concurrent reads from all reaching this point.
        # ────────────────────────────────────────────────────────────────────
        time_module.sleep(0.02)   # widen the window to make the race observable

        # Step 2: Write — insert WITHOUT holding any lock from Step 1
        _c.execute(text("""
            INSERT INTO naive_bookings (room_code, booked_from, booked_until, thread_id)
            VALUES ('OR-1', :start_dt, :end_dt, :tid)
        """), {'start_dt': _start_dt, 'end_dt': _end_dt, 'tid': thread_idx})
        _c.commit()
        return ('success', thread_idx)

NAIVE_THREADS  = 10
_naive_results = []
_naive_lock    = _threading.Lock()
_naive_barrier = _threading.Barrier(NAIVE_THREADS)

def _run_naive(idx):
    _naive_barrier.wait()   # synchronize all threads to start simultaneously
    r = naive_book(idx)
    with _naive_lock:
        _naive_results.append(r)

_naive_threads = [_threading.Thread(target=_run_naive, args=(i,)) for i in range(NAIVE_THREADS)]
for _t in _naive_threads: _t.start()
for _t in _naive_threads: _t.join()

_naive_successes = [r for r in _naive_results if r[0] == 'success']
_naive_conflicts = [r for r in _naive_results if r[0] == 'conflict']

with engine.connect() as _c:
    _naive_row_count = _c.execute(text(
        "SELECT COUNT(*) FROM naive_bookings WHERE room_code = 'OR-1'"
    )).scalar()

print(f"❌  PROBLEM — Naive (unprotected) booking results:")
print(f"    Threads reporting 'success'             : {len(_naive_successes)}")
print(f"    Threads seeing conflict                  : {len(_naive_conflicts)}")
print(f"    Rows in naive_bookings (no GIST/UNIQUE) : {_naive_row_count}")

if _naive_row_count > 1:
    print(f"\n    ⚠️  DATA CORRUPTION: {_naive_row_count} overlapping bookings exist for OR-1!")
    print(f"       Each row = a thread that passed the 'room is free' check simultaneously.")
    print(f"       This is the race condition — and exactly what SELECT FOR UPDATE prevents.")
else:
    print(f"\n    (Race window timing varies across machines. Re-run to consistently observe multiple rows.)")

# Show actual rows — visual proof of data corruption
with engine.connect() as _c:
    _corrupt_rows = _c.execute(text("""
        SELECT thread_id, room_code,
               (booked_from  AT TIME ZONE 'UTC')::text AS booked_from,
               (booked_until AT TIME ZONE 'UTC')::text AS booked_until
        FROM naive_bookings
        WHERE room_code = 'OR-1'
        ORDER BY thread_id
    """)).fetchall()

if _corrupt_rows:
    _dt = Table(
        title=f"naive_bookings — {len(_corrupt_rows)} row(s) for OR-1 slot  ← DATA CORRUPTION",
        show_header=True, header_style="bold red"
    )
    _dt.add_column("thread_id", style="dim")
    _dt.add_column("room_code", style="cyan")
    _dt.add_column("booked_from",  style="yellow")
    _dt.add_column("booked_until", style="yellow")
    for _row in _corrupt_rows:
        _dt.add_row(str(_row[0]), _row[1], _row[2][:19], _row[3][:19])
    console.print(_dt)
    print(f"  ↑ All {len(_corrupt_rows)} rows share the same time slot — multiple threads booked OR-1 simultaneously")

# Cleanup — drop the demo table after reading results
with engine.connect() as _conn:
    _conn.execute(text("DROP TABLE IF EXISTS naive_bookings"))
    _conn.commit()

❌  PROBLEM — Naive (unprotected) booking results:
    Threads reporting 'success'             : 10
    Threads seeing conflict                  : 0
    Rows in naive_bookings (no GIST/UNIQUE) : 10

    ⚠️  DATA CORRUPTION: 10 overlapping bookings exist for OR-1!
       Each row = a thread that passed the 'room is free' check simultaneously.
       This is the race condition — and exactly what SELECT FOR UPDATE prevents.


     naive_bookings — 10 row(s) for OR-1 slot  ← DATA CORRUPTION     
┏━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ thread_id ┃ room_code ┃ booked_from         ┃ booked_until        ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ 0         │ OR-1      │ 2026-04-03 08:00:00 │ 2026-04-03 10:00:00 │
│ 1         │ OR-1      │ 2026-04-03 08:00:00 │ 2026-04-03 10:00:00 │
│ 2         │ OR-1      │ 2026-04-03 08:00:00 │ 2026-04-03 10:00:00 │
│ 3         │ OR-1      │ 2026-04-03 08:00:00 │ 2026-04-03 10:00:00 │
│ 4         │ OR-1      │ 2026-04-03 08:00:00 │ 2026-04-03 10:00:00 │
│ 5         │ OR-1      │ 2026-04-03 08:00:00 │ 2026-04-03 10:00:00 │
│ 6         │ OR-1      │ 2026-04-03 08:00:00 │ 2026-04-03 10:00:00 │
│ 7         │ OR-1      │ 2026-04-03 08:00:00 │ 2026-04-03 10:00:00 │
│ 8         │ OR-1      │ 2026-04-03 08:00:00 │ 2026-04-03 10:00:00 │
│ 9         │ OR-1      │ 2026-04-03 08:00:00 │ 2026-04-03 10:00:00 │
└───────────┴───────────┴─────────────────────┴─────────────────────┘

  ↑ All 10 rows share the same time slot — multiple threads booked OR-1 simultaneously


## Test 1 — Race Condition (Part B: The Solution — SELECT FOR UPDATE)

The real `create_appointment()` uses **`SELECT FOR UPDATE`** — an exclusive row-level lock on the Room row. This makes the check-and-insert atomic: no other transaction can read or write the locked row until the first transaction commits.

```
Thread A: SELECT room FOR UPDATE  → acquires lock (others BLOCKED here)
Thread B: SELECT room FOR UPDATE  → BLOCKED, waiting for Thread A
Thread A: INSERT reservation → COMMIT → lock released
Thread B: lock granted, re-reads → sees Thread A's reservation → RoomConflictError
Result  : exactly 1 booking, clean predictable error for all other threads
```

`threading.Barrier` forces all 50 threads to start simultaneously.

In [21]:
NUM_THREADS = 50

results = []
results_lock = threading.Lock()
barrier = threading.Barrier(NUM_THREADS)

test_start_time = None 

def thread_book(thread_idx: int, case_id):
    global test_start_time
    
    # All threads wait here until all are ready
    barrier.wait()
    
    thread_start = time_module.perf_counter()
    thread_ts = datetime.now(timezone.utc)
    
    try:
        with SessionLocal() as s:
            appt = create_appointment(
                s,
                case_id=case_id,
                room_id=room_id,
                scheduled_date=ISOLATION_DATE,
                start_time=SLOT_START,
                end_time=SLOT_END,
                staff_items=[
                    StaffItem(surgeon_id, 'SURGEON'),
                    StaffItem(anaest_id, 'ANAESTHESIOLOGIST'),
                ],
                confirmed_by=surgeon_id,
            )
            s.commit()
        
        elapsed = (time_module.perf_counter() - thread_start) * 1000
        with results_lock:
            results.append({
                'thread': thread_idx,
                'outcome': 'SUCCESS',
                'appointment_id': str(appt.appointment_id),
                'timestamp': thread_ts.isoformat(),
                'elapsed_ms': elapsed,
            })
    
    except RoomConflictError as e:
        elapsed = (time_module.perf_counter() - thread_start) * 1000
        with results_lock:
            results.append({
                'thread': thread_idx,
                'outcome': 'CONFLICT',
                'error': str(e),
                'timestamp': thread_ts.isoformat(),
                'elapsed_ms': elapsed,
            })
    
    except Exception as e:
        elapsed = (time_module.perf_counter() - thread_start) * 1000
        with results_lock:
            results.append({
                'thread': thread_idx,
                'outcome': 'ERROR',
                'error': str(e),
                'timestamp': thread_ts.isoformat(),
                'elapsed_ms': elapsed,
            })

print(f"Launching {NUM_THREADS} threads all targeting OR-1 at {SLOT_START}–{SLOT_END} on {ISOLATION_DATE}")
print(f"Barrier synchronization ensures all threads start simultaneously.\n")

threads = [
    threading.Thread(target=thread_book, args=(i, case_ids[i]))
    for i in range(NUM_THREADS)
]

race_start = time_module.perf_counter()
for t in threads:
    t.start()
for t in threads:
    t.join()
race_duration = (time_module.perf_counter() - race_start) * 1000

print(f"All {NUM_THREADS} threads completed in {race_duration:.1f}ms")

Launching 50 threads all targeting OR-1 at 08:00:00–10:00:00 on 2026-04-03
Barrier synchronization ensures all threads start simultaneously.

All 50 threads completed in 184.4ms


In [22]:
# Analyze results
successes = [r for r in results if r['outcome'] == 'SUCCESS']
conflicts = [r for r in results if r['outcome'] == 'CONFLICT']
errors = [r for r in results if r['outcome'] == 'ERROR']

print(f"Results Summary:")
print(f"  SUCCESS:  {len(successes)}  (expected: 1)")
print(f"  CONFLICT: {len(conflicts)}  (expected: {NUM_THREADS - 1})")
print(f"  ERROR:    {len(errors)}   (expected: 0)")
print()

# Show all results sorted by timestamp
all_results = sorted(results, key=lambda x: x['elapsed_ms'])

t_table = Table(title=f"All {NUM_THREADS} Thread Results", show_header=True, header_style="bold")
t_table.add_column("Thread", style="dim")
t_table.add_column("Outcome", style="yellow")
t_table.add_column("Elapsed (ms)")
t_table.add_column("Detail", style="cyan", no_wrap=False)

for r in all_results:
    outcome_style = "green" if r['outcome'] == 'SUCCESS' else "red" if r['outcome'] == 'ERROR' else ""
    detail = r.get('appointment_id', r.get('error', ''))[:60]
    t_table.add_row(
        str(r['thread']),
        f"[{outcome_style}]{r['outcome']}[/{outcome_style}]" if outcome_style else r['outcome'],
        f"{r['elapsed_ms']:.1f}",
        detail
    )

console.print(t_table)

Results Summary:
  SUCCESS:  1  (expected: 1)
  CONFLICT: 49  (expected: 49)
  ERROR:    0   (expected: 0)



                                       All 50 Thread Results                                       
┏━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Thread ┃ Outcome  ┃ Elapsed (ms) ┃ Detail                                                       ┃
┡━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 4      │ CONFLICT │ 83.7         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 12     │ CONFLICT │ 84.6         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 27     │ CONFLICT │ 85.4         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 24     │ CONFLICT │ 85.7         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 20     │ CONFLICT │ 86.5         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 3      │ CONFLICT │ 86.6         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 21     │ CONFLICT │ 87.3         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 35     │ CONFLICT │ 88.2         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 36     │ CONFLICT │ 89.3         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 38     │ CONFLICT │ 93.6         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 45     │ CONFLICT │ 94.0         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 14     │ CONFLICT │ 94.2         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 22     │ CONFLICT │ 94.4         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 30     │ CONFLICT │ 94.4         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 15     │ CONFLICT │ 95.0         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 28     │ CONFLICT │ 95.5         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 16     │ CONFLICT │ 96.3         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 40     │ CONFLICT │ 97.1         │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 39     │ CONFLICT │ 100.4        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 25     │ SUCCESS  │ 101.5        │ 47b0b264-007f-4c20-a463-9039c227b05e                         │
│ 19     │ CONFLICT │ 103.6        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 26     │ CONFLICT │ 107.1        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 11     │ CONFLICT │ 107.9        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 1      │ CONFLICT │ 108.2        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 8      │ CONFLICT │ 108.5        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 7      │ CONFLICT │ 109.7        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 48     │ CONFLICT │ 110.1        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 17     │ CONFLICT │ 110.4        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 29     │ CONFLICT │ 114.2        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 43     │ CONFLICT │ 114.7        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 33     │ CONFLICT │ 116.8        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 47     │ CONFLICT │ 117.0        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 9      │ CONFLICT │ 119.9        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 41     │ CONFLICT │ 121.0        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 6      │ CONFLICT │ 121.6        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │
│ 44     │ CONFLICT │ 129.2        │ Room OR-1 is already booked from 08:00 to 10:00 on 2026-04-0 │


In [23]:
# Database verification
with engine.connect() as conn:
    active_reservations = conn.execute(text("""
        SELECT COUNT(*) FROM room_reservations rr
        JOIN appointments a ON a.appointment_id = rr.appointment_id
        WHERE rr.room_id = :room_id
          AND a.scheduled_date = :date
          AND rr.status NOT IN ('RELEASED', 'COMPLETED')
    """), {'room_id': str(room_id), 'date': ISOLATION_DATE}).scalar()

print(f"\nDatabase verification:")
print(f"  Active reservations for OR-1 on {ISOLATION_DATE}: {active_reservations}")
print(f"  (Expected: 1 — only the winning thread's reservation exists)")

# Show the actual room_reservations row — visual proof of exactly 1 clean entry
with engine.connect() as conn:
    _real_rows = conn.execute(text("""
        SELECT r.room_code, rr.status,
               (rr.reservation_start AT TIME ZONE 'UTC')::text AS res_start,
               (rr.reservation_end   AT TIME ZONE 'UTC')::text AS res_end,
               rr.reservation_id::text
        FROM room_reservations rr
        JOIN rooms r ON r.room_id = rr.room_id
        JOIN appointments a ON a.appointment_id = rr.appointment_id
        WHERE rr.room_id = :room_id
          AND a.scheduled_date = :date
          AND rr.status NOT IN ('RELEASED', 'COMPLETED')
    """), {'room_id': str(room_id), 'date': ISOLATION_DATE}).fetchall()

_rt = Table(
    title="room_reservations — actual DB rows after SELECT FOR UPDATE test",
    show_header=True, header_style="bold green"
)
_rt.add_column("room_code", style="cyan")
_rt.add_column("status",    style="green")
_rt.add_column("reservation_start", style="yellow")
_rt.add_column("reservation_end",   style="yellow")
_rt.add_column("reservation_id",    style="dim")
for _row in _real_rows:
    _rt.add_row(_row[0], _row[1], _row[2][:19], _row[3][:19], _row[4][:18] + "…")
console.print(_rt)
print(f"  ↑ Exactly 1 row — SELECT FOR UPDATE serialized all 50 threads, only 1 commit succeeded")

# Assertions
assert len(successes) == 1, f"FAIL: Expected 1 success, got {len(successes)}"
assert len(conflicts) == NUM_THREADS - 1, f"FAIL: Expected {NUM_THREADS-1} conflicts, got {len(conflicts)}"
assert len(errors) == 0, f"FAIL: Unexpected errors: {errors}"
assert active_reservations == 1, f"FAIL: Expected 1 active reservation, got {active_reservations}"

print(f"\n✅ All assertions passed:")
print(f"   exactly 1 thread succeeded")
print(f"   exactly {NUM_THREADS - 1} threads got RoomConflictError")
print(f"   exactly 1 active reservation in the database")
print(f"\nPROOF: SELECT FOR UPDATE serialized all {NUM_THREADS} threads.")
print(f"       PostgreSQL row-level locking guarantees zero double-bookings.")


Database verification:
  Active reservations for OR-1 on 2026-04-03: 1
  (Expected: 1 — only the winning thread's reservation exists)


              room_reservations — actual DB rows after SELECT FOR UPDATE test              
┏━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ room_code ┃ status    ┃ reservation_start   ┃ reservation_end     ┃ reservation_id      ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ OR-1      │ CONFIRMED │ 2026-04-03 08:00:00 │ 2026-04-03 10:00:00 │ 1970c580-bfa9-4a0e… │
└───────────┴───────────┴─────────────────────┴─────────────────────┴─────────────────────┘

  ↑ Exactly 1 row — SELECT FOR UPDATE serialized all 50 threads, only 1 commit succeeded

✅ All assertions passed:
   exactly 1 thread succeeded
   exactly 49 threads got RoomConflictError
   exactly 1 active reservation in the database

PROOF: SELECT FOR UPDATE serialized all 50 threads.
       PostgreSQL row-level locking guarantees zero double-bookings.


## Test 2 — GIST Constraint Safety Net (Database-Level Defence)

Even if application logic is **bypassed entirely** (a malicious raw SQL INSERT, a buggy script, or a direct DB connection), PostgreSQL's GIST exclusion constraint `no_room_overlap` rejects any overlapping reservation at the database level.

This is an **independent second layer** of protection — the database enforces the invariant regardless of how the INSERT arrives.

> **Note:** The INSERT below uses `gen_random_uuid()` for `appointment_id` so the UNIQUE constraint does not fire first — only the GIST exclusion constraint can block this specific insert.

In [24]:
from datetime import timezone

# Get the winning appointment ID
winning_appt_id = successes[0]['appointment_id']
print(f"Winning appointment: {winning_appt_id}")
print(f"Attempting raw SQL INSERT to double-book the same room/time...")
print(f"Using a fresh gen_random_uuid() for appointment_id so only the GIST constraint can block this.\n")

try:
    with engine.connect() as conn:
        conn.execute(text("""
            INSERT INTO room_reservations
                (reservation_id, appointment_id, room_id, status,
                 reservation_start, reservation_end, locked_at, created_at, updated_at)
            VALUES
                (gen_random_uuid(), gen_random_uuid(), :room_id, 'CONFIRMED',
                 :res_start, :res_end, NOW(), NOW(), NOW())
        """), {
            'room_id':  str(room_id),
            'res_start': datetime.combine(ISOLATION_DATE, SLOT_START, tzinfo=timezone.utc),
            'res_end':   datetime.combine(ISOLATION_DATE, SLOT_END,   tzinfo=timezone.utc),
        })
        conn.commit()
    print("❌ FAIL: GIST constraint did not prevent the double-booking!")

except IntegrityError as e:
    print(f"✅ IntegrityError raised as expected!")
    print(f"   PostgreSQL GIST exclusion constraint (no_room_overlap) rejected the INSERT.")
    error_detail = str(e.orig).split('\n')[0] if hasattr(e, 'orig') else str(e)
    print(f"   Error: {error_detail[:200]}")
except Exception as e:
    print(f"✅ Database rejected double-booking: {type(e).__name__}: {e}")

Winning appointment: 47b0b264-007f-4c20-a463-9039c227b05e
Attempting raw SQL INSERT to double-book the same room/time...
Using a fresh gen_random_uuid() for appointment_id so only the GIST constraint can block this.

✅ IntegrityError raised as expected!
   PostgreSQL GIST exclusion constraint (no_room_overlap) rejected the INSERT.
   Error: conflicting key value violates exclusion constraint "no_room_overlap"


In [25]:
print("\n" + "="*60)
print("ISOLATION TEST SUMMARY")
print("="*60)
print(f"""
Test 1: Race Condition Prevention
  {NUM_THREADS} threads attempted to book OR-1 at 08:00–10:00 simultaneously.
  Mechanism: SELECT FOR UPDATE (row-level exclusive lock on Room row)
  Result:   1 success, {NUM_THREADS-1} RoomConflictErrors, 0 double-bookings

Test 2: GIST Constraint Safety Net
  Raw SQL INSERT attempted to bypass application logic.
  Mechanism: EXCLUDE USING GIST (room_id WITH =, tstzrange(...) WITH &&)
  Result:   IntegrityError — PostgreSQL rejected the double-booking at DB level

Conclusion:
  Two independent layers prevent double-bookings:
  Layer 1 (Application): SELECT FOR UPDATE serializes concurrent transactions
  Layer 2 (Database):    GIST exclusion constraint rejects invalid INSERTs

  The system satisfies ACID Isolation requirements for concurrent OLTP workloads.
""")


ISOLATION TEST SUMMARY

Test 1: Race Condition Prevention
  50 threads attempted to book OR-1 at 08:00–10:00 simultaneously.
  Mechanism: SELECT FOR UPDATE (row-level exclusive lock on Room row)
  Result:   1 success, 49 RoomConflictErrors, 0 double-bookings

Test 2: GIST Constraint Safety Net
  Raw SQL INSERT attempted to bypass application logic.
  Mechanism: EXCLUDE USING GIST (room_id WITH =, tstzrange(...) WITH &&)
  Result:   IntegrityError — PostgreSQL rejected the double-booking at DB level

Conclusion:
  Two independent layers prevent double-bookings:
  Layer 1 (Application): SELECT FOR UPDATE serializes concurrent transactions
  Layer 2 (Database):    GIST exclusion constraint rejects invalid INSERTs

  The system satisfies ACID Isolation requirements for concurrent OLTP workloads.

